In [ ]:

import pyodbc
import ollama


# ==================================================
# 1. SQL SERVER CONNECTION
# ==================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-NB27H6F;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ==================================================
# 2. FUNCTION TO FETCH SQL DATA
# ==================================================

def get_data(query):

    cursor.execute(query)

    columns = [column[0] for column in cursor.description]

    rows = cursor.fetchall()

    result = []

    for row in rows:

        record = {}

        for i, column in enumerate(columns):
            record[column] = row[i]

        result.append(record)

    return result


# ==================================================
# 3. FORMAT SQL DATA
# ==================================================

def format_data(data):

    if not data:
        return "No records found."

    lines = []

    for record in data:

        row_data = []

        for key, value in record.items():

            row_data.append(f"{key}: {value}")

        lines.append(" | ".join(row_data))

    return "\n".join(lines)


# ==================================================
# 4. MONTHLY BUDGET VS ACTUAL COMPARISON
# ==================================================

def get_monthly_budget_comparison():

    query = """

    WITH ActualExpenses AS
    (
        SELECT

            categoryid,

            YEAR(expensedate) AS budgetyear,

            MONTH(expensedate) AS budgetmonth,

            COUNT(*) AS expensecount,

            SUM(amount) AS actualexpense

        FROM expense

        GROUP BY

            categoryid,

            YEAR(expensedate),

            MONTH(expensedate)
    ),

    BudgetData AS
    (
        SELECT

            categoryid,

            budgetyear,

            budgetmonth,

            SUM(budgetamount) AS budgetamount

        FROM expensebudget

        GROUP BY

            categoryid,

            budgetyear,

            budgetmonth
    ),

    Comparison AS
    (
        SELECT

            COALESCE(b.categoryid, a.categoryid)
                AS categoryid,

            COALESCE(b.budgetyear, a.budgetyear)
                AS budgetyear,

            COALESCE(b.budgetmonth, a.budgetmonth)
                AS budgetmonth,

            ISNULL(b.budgetamount, 0)
                AS budgetamount,

            ISNULL(a.actualexpense, 0)
                AS actualexpense,

            ISNULL(a.expensecount, 0)
                AS expensecount,

            CASE
                WHEN b.categoryid IS NULL THEN 0
                ELSE 1
            END AS hasbudget

        FROM BudgetData b

        FULL OUTER JOIN ActualExpenses a

            ON b.categoryid = a.categoryid

            AND b.budgetyear = a.budgetyear

            AND b.budgetmonth = a.budgetmonth
    )

    SELECT

        c.categoryname,

        x.budgetyear,

        x.budgetmonth,

        DATENAME(
            MONTH,
            DATEFROMPARTS(
                x.budgetyear,
                x.budgetmonth,
                1
            )
        ) AS monthname,

        x.budgetamount,

        x.actualexpense,

        x.actualexpense - x.budgetamount
            AS varianceamount,

        CASE
            WHEN x.budgetamount > 0
            THEN
                (x.actualexpense / x.budgetamount) * 100
            ELSE NULL
        END AS budgetutilization,

        x.expensecount,

        CASE

            WHEN x.hasbudget = 0
                THEN 'No Budget'

            WHEN x.actualexpense > x.budgetamount
                THEN 'Over Budget'

            WHEN x.actualexpense < x.budgetamount
                THEN 'Under Budget'

            ELSE 'On Budget'

        END AS budgetstatus

    FROM Comparison x

    LEFT JOIN expensecategory c
        ON x.categoryid = c.categoryid

    ORDER BY

        x.budgetyear,

        x.budgetmonth,

        c.categoryname;

    """

    return get_data(query)


# ==================================================
# 5. MONTHLY OVERALL BUDGET SUMMARY
# ==================================================

def get_monthly_budget_summary():

    query = """

    WITH MonthlyBudget AS
    (
        SELECT

            budgetyear,

            budgetmonth,

            SUM(budgetamount) AS totalbudget

        FROM expensebudget

        GROUP BY
            budgetyear,
            budgetmonth
    ),

    MonthlyActual AS
    (
        SELECT

            YEAR(expensedate) AS budgetyear,

            MONTH(expensedate) AS budgetmonth,

            COUNT(*) AS expensecount,

            SUM(amount) AS totalactual

        FROM expense

        GROUP BY

            YEAR(expensedate),

            MONTH(expensedate)
    ),

    MonthlyComparison AS
    (
        SELECT

            COALESCE(b.budgetyear, a.budgetyear)
                AS budgetyear,

            COALESCE(b.budgetmonth, a.budgetmonth)
                AS budgetmonth,

            ISNULL(b.totalbudget, 0)
                AS totalbudget,

            ISNULL(a.totalactual, 0)
                AS totalactual,

            ISNULL(a.expensecount, 0)
                AS expensecount

        FROM MonthlyBudget b

        FULL OUTER JOIN MonthlyActual a

            ON b.budgetyear = a.budgetyear

            AND b.budgetmonth = a.budgetmonth
    )

    SELECT

        budgetyear,

        budgetmonth,

        DATENAME(
            MONTH,
            DATEFROMPARTS(budgetyear, budgetmonth, 1)
        ) AS monthname,

        totalbudget,

        totalactual,

        totalactual - totalbudget
            AS varianceamount,

        CASE
            WHEN totalbudget > 0
            THEN (totalactual / totalbudget) * 100
            ELSE NULL
        END AS budgetutilization,

        expensecount,

        CASE

            WHEN totalbudget = 0
                THEN 'No Budget'

            WHEN totalactual > totalbudget
                THEN 'Over Budget'

            WHEN totalactual < totalbudget
                THEN 'Under Budget'

            ELSE 'On Budget'

        END AS budgetstatus

    FROM MonthlyComparison

    ORDER BY
        budgetyear,
        budgetmonth;

    """

    return get_data(query)


# ==================================================
# 6. OVER BUDGET CATEGORIES
# ==================================================

def get_over_budget_categories():

    query = """

    WITH ActualExpenses AS
    (
        SELECT

            categoryid,

            YEAR(expensedate) AS budgetyear,

            MONTH(expensedate) AS budgetmonth,

            SUM(amount) AS actualexpense

        FROM expense

        GROUP BY

            categoryid,

            YEAR(expensedate),

            MONTH(expensedate)
    )

    SELECT

        ec.categoryname,

        b.budgetyear,

        b.budgetmonth,

        b.budgetamount,

        ISNULL(a.actualexpense, 0)
            AS actualexpense,

        ISNULL(a.actualexpense, 0) - b.budgetamount
            AS overspendingamount,

        CASE
            WHEN b.budgetamount > 0
            THEN
                (a.actualexpense / b.budgetamount) * 100
            ELSE NULL
        END AS budgetutilization

    FROM expensebudget b

    INNER JOIN expensecategory ec
        ON b.categoryid = ec.categoryid

    INNER JOIN ActualExpenses a

        ON b.categoryid = a.categoryid

        AND b.budgetyear = a.budgetyear

        AND b.budgetmonth = a.budgetmonth

    WHERE a.actualexpense > b.budgetamount

    ORDER BY overspendingamount DESC;

    """

    return get_data(query)


# ==================================================
# 7. UNDER BUDGET CATEGORIES
# ==================================================

def get_under_budget_categories():

    query = """

    WITH ActualExpenses AS
    (
        SELECT

            categoryid,

            YEAR(expensedate) AS budgetyear,

            MONTH(expensedate) AS budgetmonth,

            SUM(amount) AS actualexpense

        FROM expense

        GROUP BY

            categoryid,

            YEAR(expensedate),

            MONTH(expensedate)
    )

    SELECT

        ec.categoryname,

        b.budgetyear,

        b.budgetmonth,

        b.budgetamount,

        ISNULL(a.actualexpense, 0)
            AS actualexpense,

        b.budgetamount - ISNULL(a.actualexpense, 0)
            AS remainingbudget,

        CASE
            WHEN b.budgetamount > 0
            THEN
                (ISNULL(a.actualexpense, 0) /
                 b.budgetamount) * 100
            ELSE NULL
        END AS budgetutilization

    FROM expensebudget b

    INNER JOIN expensecategory ec
        ON b.categoryid = ec.categoryid

    LEFT JOIN ActualExpenses a

        ON b.categoryid = a.categoryid

        AND b.budgetyear = a.budgetyear

        AND b.budgetmonth = a.budgetmonth

    WHERE ISNULL(a.actualexpense, 0) < b.budgetamount

    ORDER BY remainingbudget DESC;

    """

    return get_data(query)


# ==================================================
# 8. EXPENSES WITHOUT A MONTHLY BUDGET
# ==================================================

def get_unbudgeted_expenses():

    query = """

    SELECT

        ec.categoryname,

        YEAR(e.expensedate) AS expenseyear,

        MONTH(e.expensedate) AS expensemonth,

        DATENAME(MONTH, e.expensedate)
            AS monthname,

        COUNT(*) AS expensecount,

        SUM(e.amount) AS actualexpense

    FROM expense e

    LEFT JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    LEFT JOIN expensebudget b

        ON e.categoryid = b.categoryid

        AND YEAR(e.expensedate) = b.budgetyear

        AND MONTH(e.expensedate) = b.budgetmonth

    WHERE b.budgetid IS NULL

    GROUP BY

        ec.categoryname,

        YEAR(e.expensedate),

        MONTH(e.expensedate),

        DATENAME(MONTH, e.expensedate)

    ORDER BY

        expenseyear,

        expensemonth,

        actualexpense DESC;

    """

    return get_data(query)


# ==================================================
# 9. MONTHLY CATEGORY SPENDING TREND
# ==================================================

def get_category_monthly_trend():

    query = """

    WITH ActualExpenses AS
    (
        SELECT

            categoryid,

            YEAR(expensedate) AS budgetyear,

            MONTH(expensedate) AS budgetmonth,

            SUM(amount) AS actualexpense

        FROM expense

        GROUP BY

            categoryid,

            YEAR(expensedate),

            MONTH(expensedate)
    ),

    MonthlyBudget AS
    (
        SELECT

            categoryid,

            budgetyear,

            budgetmonth,

            SUM(budgetamount) AS budgetamount

        FROM expensebudget

        GROUP BY

            categoryid,

            budgetyear,

            budgetmonth
    )

    SELECT

        ec.categoryname,

        COALESCE(b.budgetyear, a.budgetyear)
            AS budgetyear,

        COALESCE(b.budgetmonth, a.budgetmonth)
            AS budgetmonth,

        ISNULL(b.budgetamount, 0)
            AS budgetamount,

        ISNULL(a.actualexpense, 0)
            AS actualexpense,

        ISNULL(a.actualexpense, 0)
            - ISNULL(b.budgetamount, 0)
            AS varianceamount

    FROM MonthlyBudget b

    FULL OUTER JOIN ActualExpenses a

        ON b.categoryid = a.categoryid

        AND b.budgetyear = a.budgetyear

        AND b.budgetmonth = a.budgetmonth

    LEFT JOIN expensecategory ec

        ON COALESCE(b.categoryid, a.categoryid)
            = ec.categoryid

    ORDER BY

        ec.categoryname,

        budgetyear,

        budgetmonth;

    """

    return get_data(query)


# ==================================================
# 10. LOAD DATABASE INFORMATION
# ==================================================

def load_budget_data():

    print("\nLoading monthly budget information from SQL Server...")

    monthly_comparison = get_monthly_budget_comparison()

    monthly_summary = get_monthly_budget_summary()

    over_budget = get_over_budget_categories()

    under_budget = get_under_budget_categories()

    unbudgeted = get_unbudgeted_expenses()

    category_trend = get_category_monthly_trend()

    all_data = f"""

    MONTHLY BUDGET VS ACTUAL COMPARISON
    ----------------------------------
    {format_data(monthly_comparison)}


    MONTHLY OVERALL BUDGET SUMMARY
    ------------------------------
    {format_data(monthly_summary)}


    OVER BUDGET CATEGORIES
    ----------------------
    {format_data(over_budget)}


    UNDER BUDGET CATEGORIES
    -----------------------
    {format_data(under_budget)}


    EXPENSES WITHOUT A MONTHLY BUDGET
    ---------------------------------
    {format_data(unbudgeted)}


    CATEGORY MONTHLY SPENDING TREND
    -------------------------------
    {format_data(category_trend)}

    """

    return all_data


# ==================================================
# 11. MONTHLY BUDGET COMPARISON AGENT
# ==================================================

def monthly_budget_comparison_agent(user_question):

    all_data = load_budget_data()

    system_prompt = """

    You are an expert Monthly Budget Comparison Agent.

    You analyse monthly budgets and actual expenses
    using SQL Server accounting data.

    YOUR RESPONSIBILITIES:

    1. Compare monthly budget with actual expenses.
    2. Analyse category-wise budget performance.
    3. Identify over-budget categories.
    4. Identify under-budget categories.
    5. Calculate budget variance.
    6. Calculate budget utilisation percentage.
    7. Identify expenses without an allocated budget.
    8. Analyse monthly expense trends.
    9. Compare different months and years.
    10. Identify recurring overspending.
    11. Identify categories requiring budget revision.
    12. Recommend cost-control measures.

    IMPORTANT ACCOUNTING RULES:

    - Budget comes from the expensebudget table.
    - Actual expense comes from the expense table.
    - Do not use payment amounts as actual expenses.
    - Do not add payments to expense amounts.
    - Variance = Actual Expense - Budget.
    - Positive variance means overspending.
    - Negative variance means underspending.
    - Budget utilisation =
      (Actual Expense / Budget) * 100.
    - If no budget exists, report actual expenses
      as unbudgeted expenses.
    - Do not calculate utilisation when budget is zero.
    - Never invent financial figures.
    - Use only the supplied SQL Server data.
    - Clearly distinguish missing budget from zero spending.
    - Compare only matching category, month and year.
    - Do not compare different months as if they were
      the same budget period.
    - Show financial figures to two decimal places.
    - Give practical cost-control recommendations.
    - Use simple professional business language.

    WHEN ANSWERING, USE THESE SECTIONS
    WHERE APPROPRIATE:

    1. Executive Summary
    2. Monthly Budget Position
    3. Category-wise Comparison
    4. Overspending Analysis
    5. Underspending Analysis
    6. Unbudgeted Expenses
    7. Monthly Spending Trends
    8. Business Risks
    9. Recommendations

    Answer the user's actual question.
    Do not unnecessarily repeat every analysis section.

    """

    user_prompt = f"""

    DATABASE INFORMATION:

    {all_data}


    USER QUESTION:

    {user_question}


    Analyse the database information and answer
    the user's question.

    """

    response = ollama.chat(

        model="llama3.2",

        messages=[

            {
                "role": "system",
                "content": system_prompt
            },

            {
                "role": "user",
                "content": user_prompt
            }

        ]

    )

    return response["message"]["content"]


# ==================================================
# 12. INTERACTIVE USER INTERFACE
# ==================================================

print("\n======================================")

print("   MONTHLY BUDGET COMPARISON AGENT")

print("======================================")

print("Database: orderssystem")

print("AI Model: llama3.2")

print("\nType 'exit' to close the agent.")

while True:

    question = input(
        "\nAsk your monthly budget question: "
    )

    if question.lower().strip() == "exit":

        break

    if not question.strip():

        print("Please enter a question.")

        continue

    try:

        answer = monthly_budget_comparison_agent(question)

        print("\nAI MONTHLY BUDGET ANALYSIS")

        print("--------------------------------------")

        print(answer)

        print("--------------------------------------")

    except Exception as e:

        print("\nError:", e)


# ==================================================
# 13. CLOSE DATABASE CONNECTION
# ==================================================

cursor.close()

connection.close()

print("\nMonthly Budget Comparison Agent stopped.")



   MONTHLY BUDGET COMPARISON AGENT
Database: orderssystem
AI Model: llama3.2

Type 'exit' to close the agent.



Ask your monthly budget question:  compare monthly budget with actual expenses



Loading monthly budget information from SQL Server...
